In [1]:
%pip install interpret==0.7.8

  Using cached interpret-0.7.8-py3-none-any.whl.metadata (1.2 kB)
  Using cached interpret_core-0.7.8-py3-none-any.whl.metadata (2.9 kB)
  Using cached salib-1.6.0-py3-none-any.whl.metadata (12 kB)
  Using cached shap-0.54.1-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (23 kB)
  Using cached aplr-10.27.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.1 kB)
  Using cached dash_cytoscape-1.0.2-py3-none-any.whl
  Using cached gevent-26.9.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (14 kB)
  Using cached flask-3.0.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached werkzeug-3.0.6-py3-none-any.whl.metadata (3.7 kB)
  Using cached dash_html_components-2.0.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached dash_core_components-2.0.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached dash_table-5.0.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached zope_event-6.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached zope_interface-8.7-cp312-cp3

In [1]:
import mlflow
import interpret
import sklearn
import pandas as pd
import boto3

print("MLflow:", mlflow.__version__)
print("InterpretML:", interpret.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Pandas:", pd.__version__)
print("AWS Region:", boto3.Session().region_name)

MLflow: 3.16.1
InterpretML: 0.7.8
Scikit-learn: 1.7.2
Pandas: 2.3.3
AWS Region: ap-southeast-2


In [3]:
import os
import mlflow

# Store MLflow database in the current notebook directory
DB_PATH = os.path.join(os.getcwd(), "mlflow.db")

# Configure SQLite tracking backend
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Create or retrieve experiment
EXPERIMENT_NAME = "Heart-Disease-Prediction"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow Version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Database Path:", DB_PATH)
print("Experiment Name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

2026/10/31 07:40:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/31 07:40:47 INFO mlflow.store.db.utils: Updating database tables
2026/10/31 07:40:49 INFO mlflow.tracking.fluent: Experiment with name 'Heart-Disease-Prediction' does not exist. Creating a new experiment.


MLflow Version: 3.16.1
Tracking URI: sqlite:////home/sagemaker-user/mlflow.db
Database Path: /home/sagemaker-user/mlflow.db
Experiment Name: Heart-Disease-Prediction
Experiment ID: 1


In [4]:
import os
import boto3
import pandas as pd

BUCKET_NAME = "heart-disease-ebm-app-2026-765807561258-ap-southeast-2-an"
S3_KEY = "data/raw/heart_disease_reference.csv"

LOCAL_DIR = "data/raw"
LOCAL_PATH = os.path.join(LOCAL_DIR, "heart_disease_reference.csv")

os.makedirs(LOCAL_DIR, exist_ok=True)

s3 = boto3.client("s3", region_name="ap-southeast-2")

s3.download_file(BUCKET_NAME, S3_KEY, LOCAL_PATH)

df = pd.read_csv(LOCAL_PATH)

print("Dataset downloaded successfully")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Dataset downloaded successfully
Shape: (270, 14)

Columns:
['age', 'sex', 'chest-pain', 'rest-bp', 'serum-chol', 'fasting-blood-sugar', 'electrocardiographic', 'max-heart-rate', 'angina', 'oldpeak', 'slope', 'major-vessels', 'thal', 'target']

First 5 rows:


,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal,target
0,70.0,1.0,4.0,130.0,322.0,0.0,2.0,109.0,0.0,2.4,2.0,3.0,3.0,1
1,67.0,0.0,3.0,115.0,564.0,0.0,2.0,160.0,0.0,1.6,2.0,0.0,7.0,0
2,57.0,1.0,2.0,124.0,261.0,0.0,0.0,141.0,0.0,0.3,1.0,0.0,7.0,1
3,64.0,1.0,4.0,128.0,263.0,0.0,0.0,105.0,1.0,0.2,2.0,1.0,7.0,0
4,74.0,0.0,2.0,120.0,269.0,0.0,2.0,121.0,1.0,0.2,1.0,1.0,3.0,0


In [5]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# Check target distribution
print("\nTarget distribution:")
print(df["target"].value_counts())

# Check data types
print("\nData types:")
print(df.dtypes)

Missing values per column:
age                     0
sex                     0
chest-pain              0
rest-bp                 0
serum-chol              0
fasting-blood-sugar     0
electrocardiographic    0
max-heart-rate          0
angina                  0
oldpeak                 0
slope                   0
major-vessels           0
thal                    0
target                  0
dtype: int64

Duplicate rows: 0

Target distribution:
target
0    150
1    120
Name: count, dtype: int64

Data types:
age                     float64
sex                     float64
chest-pain              float64
rest-bp                 float64
serum-chol              float64
fasting-blood-sugar     float64
electrocardiographic    float64
max-heart-rate          float64
angina                  float64
oldpeak                 float64
slope                   float64
major-vessels           float64
thal                    float64
target                    int64
dtype: object


In [6]:
from sklearn.model_selection import train_test_split

# Define input features and target
X = df.drop(columns=["target"])
y = df["target"]

# Feature categories
categorical_features = [
    "sex", "chest-pain", "fasting-blood-sugar",
    "electrocardiographic", "angina", "slope",
    "major-vessels", "thal"
]

continuous_features = [
    "age", "rest-bp", "serum-chol",
    "max-heart-rate", "oldpeak"
]

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training features: (216, 13)
Testing features: (54, 13)

Training target distribution:
target
0    120
1     96
Name: count, dtype: int64

Testing target distribution:
target
0    30
1    24
Name: count, dtype: int64


In [7]:
import mlflow
import mlflow.sklearn

from interpret.glassbox import ExplainableBoostingClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

# Cross-validation configuration
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define EBM
ebm_model = ExplainableBoostingClassifier(
    interactions=0,
    random_state=42
)

# Define Logistic Regression pipeline
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

print("EBM model configured.")
print("Logistic Regression configured.")
print("5-fold stratified cross-validation configured.")

EBM model configured.
Logistic Regression configured.
5-fold stratified cross-validation configured.


In [12]:
# Define feature types exactly as in the original notebook

feature_types = [
    "nominal" if col in categorical_features else "continuous"
    for col in X_train.columns
]

# Reconfigure EBM
ebm_model = ExplainableBoostingClassifier(
    feature_types=feature_types,
    random_state=42,
    interactions=0
)

# Reconfigure Logistic Regression
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("EBM configuration restored.")
print("Logistic Regression configuration restored.")
print("\nFeature types:", feature_types)

EBM configuration restored.
Logistic Regression configuration restored.

Feature types: ['continuous', 'nominal', 'nominal', 'continuous', 'continuous', 'nominal', 'nominal', 'continuous', 'nominal', 'continuous', 'nominal', 'nominal', 'nominal']


In [15]:

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

models = {
    "EBM": ebm_model,
    "Logistic_Regression": lr_model
}

cv_results = {}

for model_name, model in models.items():

    print(f"\nEvaluating {model_name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1
    )

    metrics = {
        metric: float(scores[f"test_{metric}"].mean())
        for metric in scoring
    }

    cv_results[model_name] = metrics

    with mlflow.start_run(run_name=f"{model_name}_CV_Corrected"):
        mlflow.log_param("model", model_name)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("random_state", 42)
        mlflow.log_metrics(metrics)

    print(metrics)

cv_results_df = pd.DataFrame(cv_results).T

print("\nCross-validation comparison:")
display(cv_results_df)



Evaluating EBM...
{'accuracy': 0.8423890063424946, 'precision': 0.8394938753533449, 'recall': 0.8126315789473683, 'f1': 0.8219955293639505, 'roc_auc': 0.9047368421052632}

Evaluating Logistic_Regression...
{'accuracy': 0.8192389006342495, 'precision': 0.8135461760461761, 'recall': 0.781578947368421, 'f1': 0.7931239601971309, 'roc_auc': 0.8976096491228069}

Cross-validation comparison:


,accuracy,precision,recall,f1,roc_auc
EBM,0.842389,0.839494,0.812632,0.821996,0.904737
Logistic_Regression,0.819239,0.813546,0.781579,0.793124,0.897610


In [16]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

final_models = {
    "EBM": clone(ebm_model),
    "Logistic_Regression": clone(lr_model)
}

test_results = {}
trained_models = {}

for model_name, model in final_models.items():

    print(f"\nTraining {model_name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob)
    }

    test_results[model_name] = metrics
    trained_models[model_name] = model

    cm = confusion_matrix(y_test, y_pred)

    print("Test Metrics:", metrics)
    print("Confusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Log final test results in MLflow
    with mlflow.start_run(run_name=f"{model_name}_Test_Corrected"):

        mlflow.log_param("model", model_name)
        mlflow.log_param("dataset", "UCI_Statlog_Heart_Disease")
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("random_state", 42)

        mlflow.log_metrics({
            f"test_{key}": float(value)
            for key, value in metrics.items()
        })

        mlflow.log_metric("test_true_negative", int(cm[0, 0]))
        mlflow.log_metric("test_false_positive", int(cm[0, 1]))
        mlflow.log_metric("test_false_negative", int(cm[1, 0]))
        mlflow.log_metric("test_true_positive", int(cm[1, 1]))

print("\nFinal Test Comparison:")
display(pd.DataFrame(test_results).T)



Training EBM...
Test Metrics: {'accuracy': 0.8703703703703703, 'precision': 0.8148148148148148, 'recall': 0.9166666666666666, 'f1': 0.8627450980392157, 'roc_auc': 0.9083333333333333}
Confusion Matrix:
[[25  5]
 [ 2 22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.83      0.88        30
           1       0.81      0.92      0.86        24

    accuracy                           0.87        54
   macro avg       0.87      0.88      0.87        54
weighted avg       0.88      0.87      0.87        54


Training Logistic_Regression...
Test Metrics: {'accuracy': 0.8518518518518519, 'precision': 0.7857142857142857, 'recall': 0.9166666666666666, 'f1': 0.8461538461538461, 'roc_auc': 0.898611111111111}
Confusion Matrix:
[[24  6]
 [ 2 22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.80      0.86        30
           1       0.79      0.92      0.85        24

    

,accuracy,precision,recall,f1,roc_auc
EBM,0.870370,0.814815,0.916667,0.862745,0.908333
Logistic_Regression,0.851852,0.785714,0.916667,0.846154,0.898611


In [17]:

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

display(
    runs[
        ["tags.mlflow.runName", "status",
         "metrics.accuracy",
         "metrics.test_accuracy"]
    ].head(10)
)


,tags.mlflow.runName,status,metrics.accuracy,metrics.test_accuracy
0,Logistic_Regression_Test_Corrected,FINISHED,NaN,0.851852
1,EBM_Test_Corrected,FINISHED,NaN,0.870370
2,Logistic_Regression_CV_Corrected,FINISHED,0.819239,NaN
3,EBM_CV_Corrected,FINISHED,0.842389,NaN
4,Logistic_Regression_CV,FINISHED,0.819239,NaN
5,EBM_CV,FINISHED,0.842389,NaN
6,Logistic_Regression_CV,FINISHED,0.819239,NaN
7,EBM_CV,FINISHED,0.847040,NaN


In [19]:

import mlflow
import mlflow.sklearn

all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

for model_name, model in trained_models.items():

    run_name = f"{model_name}_Test_Corrected"

    matching_runs = all_runs[
        all_runs["tags.mlflow.runName"] == run_name
    ]

    if matching_runs.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching_runs.iloc[0]["run_id"]

    with mlflow.start_run(run_id=run_id):

        if model_name == "EBM":
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model",
                skops_trusted_types=[
                    "interpret.glassbox._ebm._ebm.ExplainableBoostingClassifier"
                ]
            )
        else:
            mlflow.sklearn.log_model(
                sk_model=model,
                name="model"
            )

    print(f"{model_name} model saved to MLflow.")


EBM model saved to MLflow.
Logistic_Regression model saved to MLflow.


In [21]:
import time
import numpy as np
import pandas as pd

comparable_latency_results = {}

for model_name, model in trained_models.items():

    sample = X_test.iloc[[0]]

    # Warm-up
    for _ in range(20):
        model.predict_proba(sample)

    latencies = []

    # Measure only predict_proba()
    for _ in range(500):
        start = time.perf_counter_ns()

        model.predict_proba(sample)

        end = time.perf_counter_ns()

        latencies.append((end - start) / 1_000_000)

    comparable_latency_results[model_name] = {
        "mean_ms": float(np.mean(latencies)),
        "median_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "min_ms": float(np.min(latencies)),
        "max_ms": float(np.max(latencies)),
        "requests": 500,
        "warmup": 20
    }

latency_comparison_df = pd.DataFrame(
    comparable_latency_results
).T

display(latency_comparison_df.round(4))

,mean_ms,median_ms,p95_ms,p99_ms,min_ms,max_ms,requests,warmup
EBM,1.0164,0.9153,1.3862,1.5174,0.7362,3.0460,500.0,20.0
Logistic_Regression,1.0469,0.9754,1.4108,1.6232,0.8178,4.8807,500.0,20.0


In [22]:
import mlflow

for model_name, metrics in comparable_latency_results.items():

    run_name = f"{model_name}_Test_Corrected"

    all_runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    matching_runs = all_runs[
        all_runs["tags.mlflow.runName"] == run_name
    ]

    if matching_runs.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching_runs.iloc[0]["run_id"]

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metrics({
            f"comparable_latency_{key}": float(value)
            for key, value in metrics.items()
            if key not in ["requests", "warmup"]
        })

        mlflow.set_tag("latency_benchmark_method", "predict_proba_only")
        mlflow.set_tag("latency_warmup_iterations", "20")
        mlflow.set_tag("latency_measurement_iterations", "500")

    print(f"{model_name}: comparable latency metrics logged.")

EBM: comparable latency metrics logged.
Logistic_Regression: comparable latency metrics logged.


In [23]:
import mlflow

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]
)

verification = runs[
    runs["tags.mlflow.runName"].isin([
        "EBM_Test_Corrected",
        "Logistic_Regression_Test_Corrected"
    ])
]

columns = [
    "tags.mlflow.runName",
    "status",
    "metrics.test_accuracy",
    "metrics.comparable_latency_mean_ms",
    "metrics.comparable_latency_median_ms",
    "metrics.comparable_latency_p95_ms",
    "metrics.comparable_latency_p99_ms"
]

display(verification[columns])

,tags.mlflow.runName,status,metrics.test_accuracy,metrics.comparable_latency_mean_ms,metrics.comparable_latency_median_ms,metrics.comparable_latency_p95_ms,metrics.comparable_latency_p99_ms
0,Logistic_Regression_Test_Corrected,FINISHED,0.851852,1.046869,0.975362,1.410805,1.623238
1,EBM_Test_Corrected,FINISHED,0.870370,1.016391,0.915289,1.386234,1.517372


In [24]:
import boto3

s3 = boto3.client("s3")

BUCKET_NAME = "heart-disease-ebm-app-2026-765807561258-ap-southeast-2-an"

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="artifacts/",
    MaxKeys=30
)

for obj in response.get("Contents", []):
    print(obj["Key"], round(obj["Size"] / 1024, 2), "KB")

artifacts/initial/categorical_features.json 0.11 KB
artifacts/initial/ebm_model.pkl 60.06 KB
artifacts/initial/feature_columns.json 0.17 KB
artifacts/initial/local_latency.csv 0.26 KB
artifacts/initial/local_latency.json 0.54 KB
artifacts/initial/logistic_model.pkl 2.13 KB
artifacts/initial/metadata.json 0.23 KB
artifacts/initial/test_metrics.csv 0.25 KB


In [25]:
import os
import mlflow

EXPORT_DIR = "artifacts/mlflow_corrected"
os.makedirs(EXPORT_DIR, exist_ok=True)

for model_name in ["EBM", "Logistic_Regression"]:

    run_name = f"{model_name}_Test_Corrected"

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id]
    )

    matching = runs[
        runs["tags.mlflow.runName"] == run_name
    ]

    if matching.empty:
        print(f"Run not found: {run_name}")
        continue

    run_id = matching.iloc[0]["run_id"]

    model_uri = f"runs:/{run_id}/model"

    local_path = mlflow.artifacts.download_artifacts(
        artifact_uri=model_uri,
        dst_path=EXPORT_DIR
    )

    print(f"{model_name} exported to: {local_path}")

EBM exported to: /home/sagemaker-user/artifacts/mlflow_corrected/model


Logistic_Regression exported to: /home/sagemaker-user/artifacts/mlflow_corrected/model


In [26]:
import os
import shutil

EXPORT_DIR = "artifacts/mlflow_corrected"

ebm_source = os.path.join(EXPORT_DIR, "model")
ebm_destination = os.path.join(EXPORT_DIR, "ebm_model")

if os.path.exists(ebm_destination):
    shutil.rmtree(ebm_destination)

shutil.move(ebm_source, ebm_destination)

# Download Logistic Regression again into a separate folder
lr_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

lr_run_id = lr_runs.loc[
    lr_runs["tags.mlflow.runName"] == "Logistic_Regression_Test_Corrected",
    "run_id"
].iloc[0]

lr_destination = os.path.join(EXPORT_DIR, "logistic_model")

mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/{lr_run_id}/model",
    dst_path=lr_destination
)

print("Exported model folders:")
print(os.listdir(EXPORT_DIR))

Exported model folders:
['ebm_model', 'logistic_model']


In [27]:
import os
import json
import pandas as pd
from datetime import datetime, timezone

EXPORT_DIR = "artifacts/mlflow_corrected"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Feature information
with open(os.path.join(EXPORT_DIR, "feature_columns.json"), "w") as f:
    json.dump(list(X_train.columns), f, indent=4)

with open(os.path.join(EXPORT_DIR, "categorical_features.json"), "w") as f:
    json.dump(categorical_features, f, indent=4)

# Test metrics
test_metrics = {
    "EBM": {
        "accuracy": 0.8703703704,
        "precision": 0.8148148148,
        "recall": 0.9166666667,
        "f1": 0.8627450980,
        "roc_auc": 0.9083333333
    },
    "Logistic_Regression": {
        "accuracy": 0.8518518519,
        "precision": 0.7857142857,
        "recall": 0.9166666667,
        "f1": 0.8461538462,
        "roc_auc": 0.8986111111
    }
}

with open(os.path.join(EXPORT_DIR, "test_metrics.json"), "w") as f:
    json.dump(test_metrics, f, indent=4)

# Comparable latency metrics
with open(os.path.join(EXPORT_DIR, "latency_metrics.json"), "w") as f:
    json.dump(comparable_latency_results, f, indent=4)

# Project metadata
metadata = {
    "project": "Explainable and Drift-Aware MLOps Pipeline for Heart Disease Prediction",
    "dataset": "UCI Statlog Heart Disease",
    "primary_model": "Explainable Boosting Machine",
    "baseline_model": "Logistic Regression",
    "training_environment": "Amazon SageMaker Studio",
    "tracking_tool": "MLflow",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "latency_method": "predict_proba only; 20 warmups; 500 measurements",
    "note": "Educational research prototype; not for clinical diagnosis."
}

with open(os.path.join(EXPORT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)

print("Prepared files:")
print(os.listdir(EXPORT_DIR))

Prepared files:
['ebm_model', 'logistic_model', 'feature_columns.json', 'categorical_features.json', 'test_metrics.json', 'latency_metrics.json', 'metadata.json']


In [28]:
import os

BUCKET_NAME = "heart-disease-ebm-app-2026-765807561258-ap-southeast-2-an"
LOCAL_DIR = "artifacts/mlflow_corrected"
S3_PREFIX = "artifacts/mlflow_corrected/"

for root, dirs, files in os.walk(LOCAL_DIR):
    for filename in files:
        local_path = os.path.join(root, filename)

        relative_path = os.path.relpath(local_path, LOCAL_DIR)
        s3_key = S3_PREFIX + relative_path.replace("\\", "/")

        s3.upload_file(
            local_path,
            BUCKET_NAME,
            s3_key
        )

        print(f"Uploaded: {s3_key}")

print("\nUpload completed.")

Uploaded: artifacts/mlflow_corrected/feature_columns.json
Uploaded: artifacts/mlflow_corrected/categorical_features.json
Uploaded: artifacts/mlflow_corrected/test_metrics.json
Uploaded: artifacts/mlflow_corrected/latency_metrics.json
Uploaded: artifacts/mlflow_corrected/metadata.json
Uploaded: artifacts/mlflow_corrected/ebm_model/MLmodel
Uploaded: artifacts/mlflow_corrected/ebm_model/model.skops
Uploaded: artifacts/mlflow_corrected/ebm_model/conda.yaml
Uploaded: artifacts/mlflow_corrected/ebm_model/requirements.txt
Uploaded: artifacts/mlflow_corrected/ebm_model/python_env.yaml
Uploaded: artifacts/mlflow_corrected/ebm_model/registered_model_meta
Uploaded: artifacts/mlflow_corrected/logistic_model/model/MLmodel
Uploaded: artifacts/mlflow_corrected/logistic_model/model/conda.yaml
Uploaded: artifacts/mlflow_corrected/logistic_model/model/model.skops
Uploaded: artifacts/mlflow_corrected/logistic_model/model/python_env.yaml
Uploaded: artifacts/mlflow_corrected/logistic_model/model/requiremen

In [29]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="artifacts/mlflow_corrected/"
)

print("Files stored in S3:\n")

for obj in response.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.2f} KB)")

print("\nTotal files:", len(response.get("Contents", [])))

Files stored in S3:

artifacts/mlflow_corrected/categorical_features.json  (0.14 KB)
artifacts/mlflow_corrected/ebm_model/MLmodel  (0.68 KB)
artifacts/mlflow_corrected/ebm_model/conda.yaml  (0.43 KB)
artifacts/mlflow_corrected/ebm_model/model.skops  (29.64 KB)
artifacts/mlflow_corrected/ebm_model/python_env.yaml  (0.12 KB)
artifacts/mlflow_corrected/ebm_model/registered_model_meta  (0.04 KB)
artifacts/mlflow_corrected/ebm_model/requirements.txt  (0.27 KB)
artifacts/mlflow_corrected/feature_columns.json  (0.22 KB)
artifacts/mlflow_corrected/latency_metrics.json  (0.54 KB)
artifacts/mlflow_corrected/logistic_model/model/MLmodel  (0.68 KB)
artifacts/mlflow_corrected/logistic_model/model/conda.yaml  (0.43 KB)
artifacts/mlflow_corrected/logistic_model/model/model.skops  (29.64 KB)
artifacts/mlflow_corrected/logistic_model/model/python_env.yaml  (0.12 KB)
artifacts/mlflow_corrected/logistic_model/model/registered_model_meta  (0.04 KB)
artifacts/mlflow_corrected/logistic_model/model/requireme

In [30]:
%pip install evidently

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 166.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 47.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.7/584.7 kB 58.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 79.6 MB/s  0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 6.0.1
    Uninstalling plotly-6.0.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/15 [plotly]
      Successfully uninstalled plotly-6.0.1━━━━━━━━━━━━━━━━━━━━━━━  3/15 [plotly]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [evidently]15 [evidently]y]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
catboost 1.2.7 requires numpy<2.0,>=1.16.0, but you have numpy 2.5.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [31]:
import evidently
import numpy as np
import sklearn

print("Evidently:", evidently.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Evidently: 0.7.23
NumPy: 2.5.4
Scikit-learn: 1.7.2


In [32]:
import os

os.makedirs("data/monitoring", exist_ok=True)

reference_data = X_train.copy()

reference_data.to_csv(
    "data/monitoring/reference_data.csv",
    index=False
)

print("Reference dataset saved.")
print("Shape:", reference_data.shape)
print("\nMissing values:", reference_data.isnull().sum().sum())
print("\nFirst five rows:")
display(reference_data.head())

Reference dataset saved.
Shape: (216, 13)

Missing values: 0

First five rows:


,age,sex,chest-pain,rest-bp,serum-chol,fasting-blood-sugar,electrocardiographic,max-heart-rate,angina,oldpeak,slope,major-vessels,thal
78,42.0,0.0,3.0,120.0,209.0,0.0,0.0,173.0,0.0,0.0,2.0,0.0,3.0
121,54.0,1.0,4.0,122.0,286.0,0.0,2.0,116.0,1.0,3.2,2.0,2.0,3.0
27,51.0,0.0,3.0,120.0,295.0,0.0,2.0,157.0,0.0,0.6,1.0,0.0,3.0
198,69.0,0.0,1.0,140.0,239.0,0.0,0.0,151.0,0.0,1.8,1.0,2.0,3.0
218,54.0,1.0,3.0,120.0,258.0,0.0,2.0,147.0,0.0,0.4,2.0,0.0,7.0


In [33]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Normal production data
normal_data = X_train.sample(
    n=150,
    replace=True,
    random_state=42
).reset_index(drop=True)

# Drifted production data
drifted_data = normal_data.copy()

# Introduce controlled distribution changes
drifted_data["age"] += 10
drifted_data["serum-chol"] += 40
drifted_data["max-heart-rate"] -= 20
drifted_data["oldpeak"] += 1.0

# Save both datasets
normal_data.to_csv(
    "data/monitoring/normal_production_data.csv",
    index=False
)

drifted_data.to_csv(
    "data/monitoring/drifted_production_data.csv",
    index=False
)

print("Normal production shape:", normal_data.shape)
print("Drifted production shape:", drifted_data.shape)

print("\nReference means:")
display(reference_data.mean().round(2))

print("\nDrifted production means:")
display(drifted_data.mean().round(2))

Normal production shape: (150, 13)
Drifted production shape: (150, 13)

Reference means:


age                      54.36
sex                       0.69
chest-pain                3.10
rest-bp                 131.07
serum-chol              248.29
fasting-blood-sugar       0.13
electrocardiographic      1.05
max-heart-rate          150.56
angina                    0.32
oldpeak                   1.04
slope                     1.58
major-vessels             0.66
thal                      4.65
dtype: float64


Drifted production means:


age                      64.19
sex                       0.67
chest-pain                3.06
rest-bp                 129.39
serum-chol              286.85
fasting-blood-sugar       0.11
electrocardiographic      1.08
max-heart-rate          130.38
angina                    0.32
oldpeak                   2.11
slope                     1.59
major-vessels             0.77
thal                      4.69
dtype: float64

In [34]:
from evidently import Report
from evidently.presets import DataDriftPreset

# Create drift report for normal production data
normal_report = Report([
    DataDriftPreset()
])

normal_result = normal_report.run(
    reference_data=reference_data,
    current_data=normal_data
)

normal_result.save_html(
    "data/monitoring/normal_drift_report.html"
)

# Create drift report for drifted production data
drifted_report = Report([
    DataDriftPreset()
])

drifted_result = drifted_report.run(
    reference_data=reference_data,
    current_data=drifted_data
)

drifted_result.save_html(
    "data/monitoring/drifted_report.html"
)

print("Both drift reports generated successfully.")

Both drift reports generated successfully.


In [38]:
DRIFT_THRESHOLD = 0.20

normal_drift_share = normal_dict["metrics"][0]["value"]["share"]
drifted_share = drifted_dict["metrics"][0]["value"]["share"]

def drift_decision(drift_share):
    if drift_share >= DRIFT_THRESHOLD:
        return "DRIFT_ALERT"
    return "NORMAL"

normal_status = drift_decision(normal_drift_share)
drifted_status = drift_decision(drifted_share)

print("Normal production status:", normal_status)
print("Normal drift share:", round(normal_drift_share, 4))

print("\nDrifted production status:", drifted_status)
print("Drifted share:", round(drifted_share, 4))

Normal production status: NORMAL
Normal drift share: 0.0

Drifted production status: DRIFT_ALERT
Drifted share: 0.3077


In [39]:
import os
import json
from datetime import datetime, timezone

monitoring_results = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "reference_dataset": "UCI Statlog Heart Disease training data",
    "normal_production": {
        "drift_share": normal_drift_share,
        "status": normal_status
    },
    "drifted_production": {
        "drift_share": drifted_share,
        "status": drifted_status
    },
    "alert_threshold": DRIFT_THRESHOLD,
    "monitoring_tool": "Evidently",
    "evidently_version": evidently.__version__
}

with open(
    "data/monitoring/drift_monitoring_results.json",
    "w"
) as f:
    json.dump(monitoring_results, f, indent=4)

print(json.dumps(monitoring_results, indent=4))

{
    "timestamp_utc": "2026-10-31T09:26:44.701943+00:00",
    "reference_dataset": "UCI Statlog Heart Disease training data",
    "normal_production": {
        "drift_share": 0.0,
        "status": "NORMAL"
    },
    "drifted_production": {
        "drift_share": 0.3076923076923077,
        "status": "DRIFT_ALERT"
    },
    "alert_threshold": 0.2,
    "monitoring_tool": "Evidently",
    "evidently_version": "0.7.23"
}


In [40]:
import os

MONITORING_DIR = "data/monitoring"
S3_PREFIX = "artifacts/monitoring/"

for root, dirs, files in os.walk(MONITORING_DIR):
    for filename in files:
        local_path = os.path.join(root, filename)

        relative_path = os.path.relpath(
            local_path, MONITORING_DIR
        )

        s3_key = S3_PREFIX + relative_path.replace("\\", "/")

        s3.upload_file(
            local_path,
            BUCKET_NAME,
            s3_key
        )

        print("Uploaded:", s3_key)

print("\nMonitoring artifacts uploaded.")

Uploaded: artifacts/monitoring/reference_data.csv
Uploaded: artifacts/monitoring/normal_production_data.csv
Uploaded: artifacts/monitoring/drifted_production_data.csv
Uploaded: artifacts/monitoring/normal_drift_report.html
Uploaded: artifacts/monitoring/drifted_report.html
Uploaded: artifacts/monitoring/drift_monitoring_results.json

Monitoring artifacts uploaded.


In [41]:
import boto3

sts = boto3.client("sts")
print("AWS identity:")
print(sts.get_caller_identity())

for service in ["lambda", "apigateway"]:
    try:
        if service == "lambda":
            client = boto3.client("lambda")
            client.list_functions(MaxItems=1)
        else:
            client = boto3.client("apigateway")
            client.get_rest_apis(limit=1)

        print(f"{service}: Read access available")

    except Exception as e:
        print(f"{service}: {type(e).__name__}: {e}")

AWS identity:
{'UserId': 'AROA3ETNQBIVPWTO66FXV:SageMaker', 'Account': '765807561258', 'Arn': 'arn:aws:sts::765807561258:assumed-role/AmazonSageMaker-ExecutionRole-20261030T124287/SageMaker', 'ResponseMetadata': {'RequestId': '4ea0a559-74ca-4593-a7cb-0953415fc776', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '4ea0a559-74ca-4593-a7cb-0953415fc776', 'x-amz-sts-extended-request-id': 'MTphcC1zb3V0aGVhc3QtMjpTOjE3OTM0MzkwOTc1MTY6UjpOUWFvNzM2dg==', 'content-type': 'text/xml', 'content-length': '470', 'date': 'Sat, 31 Oct 2026 09:31:37 GMT'}, 'RetryAttempts': 0}}
lambda: Read access available
apigateway: Read access available


In [42]:
import boto3

iam = boto3.client("iam")

role_arn = (
    "arn:aws:iam::765807561258:role/service-role/"
    "AmazonSageMaker-ExecutionRole-20261030T124287"
)

actions = [
    "lambda:CreateFunction",
    "lambda:UpdateFunctionCode",
    "lambda:AddPermission",
    "apigateway:POST",
    "iam:PassRole",
    "logs:CreateLogGroup",
    "logs:PutLogEvents"
]

try:
    result = iam.simulate_principal_policy(
        PolicySourceArn=role_arn,
        ActionNames=actions
    )

    for item in result["EvaluationResults"]:
        print(
            item["EvalActionName"],
            ":", item["EvalDecision"]
        )

except Exception as e:
    print(type(e).__name__, e)

lambda:CreateFunction : explicitDeny
lambda:UpdateFunctionCode : explicitDeny
lambda:AddPermission : explicitDeny
apigateway:POST : explicitDeny
iam:PassRole : allowed
logs:CreateLogGroup : allowed
logs:PutLogEvents : allowed


In [43]:
import boto3

ec2 = boto3.client("ec2", region_name="ap-southeast-2")

try:
    response = ec2.describe_instances(MaxResults=5)
    print("EC2 read access available")

except Exception as e:
    print(type(e).__name__, e)

EC2 read access available


In [44]:
import boto3

ec2 = boto3.client("ec2", region_name="ap-southeast-2")

INSTANCE_ID = "i-06bfaba3b5af8078e"

response = ec2.describe_instances(
    InstanceIds=[INSTANCE_ID]
)

for reservation in response["Reservations"]:
    for instance in reservation["Instances"]:
        print("Instance ID:", instance["InstanceId"])
        print("State:", instance["State"]["Name"])
        print("Instance Type:", instance["InstanceType"])
        print("Public IP:", instance.get("PublicIpAddress"))
        print("Private IP:", instance.get("PrivateIpAddress"))
        print("Security Groups:", instance["SecurityGroups"])

Instance ID: i-06bfaba3b5af8078e
State: stopped
Instance Type: t3.micro
Public IP: None
Private IP: 172.31.41.56
Security Groups: [{'GroupId': 'sg-03dec5a171853d247', 'GroupName': 'HeartDiseaseApp-SG'}]


In [45]:
import boto3

iam = boto3.client("iam")

role_name = "HeartDiseaseEC2S3Role"

print("Attached policies:")
policies = iam.list_attached_role_policies(
    RoleName=role_name
)

for policy in policies["AttachedPolicies"]:
    print(policy["PolicyName"], policy["PolicyArn"])

print("\nInline policies:")
inline = iam.list_role_policies(RoleName=role_name)

for policy_name in inline["PolicyNames"]:
    print(policy_name)

Attached policies:
AmazonS3ReadOnlyAccess arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess

Inline policies:
HeartDiseaseCloudWatchLogsPolicy


In [46]:
import boto3
import json

iam = boto3.client("iam")

policy = iam.get_role_policy(
    RoleName="HeartDiseaseEC2S3Role",
    PolicyName="HeartDiseaseCloudWatchLogsPolicy"
)

print(json.dumps(policy["PolicyDocument"], indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "logs:CreateLogGroup",
        "logs:CreateLogStream",
        "logs:PutLogEvents",
        "logs:DescribeLogStreams"
      ],
      "Resource": "*"
    }
  ]
}


In [47]:
import mlflow

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

print(
    runs[
        runs["tags.mlflow.runName"].isin([
            "EBM_Test_Corrected",
            "Logistic_Regression_Test_Corrected"
        ])
    ][["tags.mlflow.runName", "run_id", "status"]]
)

                  tags.mlflow.runName                            run_id  \
0  Logistic_Regression_Test_Corrected  31ed2f9b61814e6389f3dc9d6f0ae501   
1                  EBM_Test_Corrected  a7fea3db74ad47c8b9d0fa85d7997747   

     status  
0  FINISHED  
1  FINISHED  


In [48]:
import mlflow

EBM_RUN_ID = "a7fea3db74ad47c8b9d0fa85d7997747"

ebm_model_uri = f"runs:/{EBM_RUN_ID}/model"

ebm_local_path = mlflow.artifacts.download_artifacts(
    artifact_uri=ebm_model_uri,
    dst_path="artifacts/verified_ebm"
)

print("Downloaded EBM artifact:", ebm_local_path)

Downloaded EBM artifact: /home/sagemaker-user/artifacts/verified_ebm/model


In [49]:
import mlflow.sklearn

verified_ebm = mlflow.sklearn.load_model(ebm_local_path)

print("Model class:", type(verified_ebm))

Model class: <class 'interpret.glassbox._ebm._ebm.ExplainableBoostingClassifier'>


In [50]:
import boto3
import os

bucket = "heart-disease-ebm-app-2026-765807561258-ap-southeast-2-an"

local_dir = ebm_local_path
s3_prefix = "artifacts/verified_ebm/model/"

s3 = boto3.client("s3", region_name="ap-southeast-2")

for root, dirs, files in os.walk(local_dir):
    for filename in files:
        local_path = os.path.join(root, filename)
        relative_path = os.path.relpath(local_path, local_dir)
        s3_key = s3_prefix + relative_path.replace("\\", "/")

        s3.upload_file(local_path, bucket, s3_key)
        print("Uploaded:", s3_key)

print("Verified EBM model uploaded successfully!")

Uploaded: artifacts/verified_ebm/model/MLmodel
Uploaded: artifacts/verified_ebm/model/conda.yaml
Uploaded: artifacts/verified_ebm/model/model.skops
Uploaded: artifacts/verified_ebm/model/requirements.txt
Uploaded: artifacts/verified_ebm/model/python_env.yaml
Uploaded: artifacts/verified_ebm/model/registered_model_meta
Verified EBM model uploaded successfully!


In [2]:
import os

db_path = "/home/sagemaker-user/mlflow.db"

print("Database exists:", os.path.exists(db_path))
if os.path.exists(db_path):
    print("Database size:", os.path.getsize(db_path), "bytes")

Database exists: True
Database size: 905216 bytes


In [3]:
import mlflow
print("MLflow version:", mlflow.__version__)

MLflow version: 3.16.1
